# Kelly vs. SBF vs. risk-neutral gamblers — a longshot lottery

A different gamble: each round, with probability $0.10$ you win $10{,}000 \times$ your bet size; with probability $0.90$ you lose the bet. Per dollar bet, the gross return is $R = 10{,}001$ (win) or $R = 0$ (lose). The arithmetic per-dollar EV is $0.1 \cdot 10{,}001 - 1 \approx 999$ — wildly positive. Each player bets a fraction $f$ of wealth each round, so wealth evolves as
$$W_{t+1} \;=\; W_t \bigl(1 + f\,(R - 1)\bigr).$$
We simulate $N = 10{,}000$ players for each of three strategies:

- **Kelly** ($f^* \approx 0.0999$) — log-utility / $\gamma = 1$ optimum on this lottery.
- **SBF** ($f^* \approx 0.5350$) — CRRA optimum at $\gamma = 0.75$ on this lottery (more than five times Kelly's bet).
- **Risk neutral** ($f = 1$) — maximizes arithmetic $E[W_T]$ by going all-in each round; one losing round wipes you out.

We play these strategies forward 100 rounds and see where the players end up."


In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(544)

In [2]:
initial_investment = 1000
T = 100         # rounds per player
N = 10_000      # players per strategy

# Lottery: P(R = 10_001) = 0.10, P(R = 0) = 0.90.
returns = np.array([10_001.0, 0.0])
probs   = np.array([0.10, 0.90])

f_kelly = 0.0999    # log-utility optimum on this lottery
f_sbf   = 0.5350    # CRRA optimum at gamma = 0.75 on this lottery
f_rn    = 1.0       # risk-neutral: all-in each round

In [3]:
# Vectorized Monte Carlo: each row is one player's T draws of the underlying asset.
draws = rng.choice(returns, size=(N, T), p=probs)

kelly_multipliers = 1 + f_kelly * (draws - 1)
sbf_multipliers   = 1 + f_sbf   * (draws - 1)
rn_multipliers    = 1 + f_rn    * (draws - 1)   # equals draws itself when f=1

kelly_final = initial_investment * np.prod(kelly_multipliers, axis=1)
sbf_final   = initial_investment * np.prod(sbf_multipliers,   axis=1)
rn_final    = initial_investment * np.prod(rn_multipliers,    axis=1)

## Expected final wealth, survival, and "above start" rates

Returns are i.i.d., so $E[W_T] = W_0 \cdot (E[\text{multiplier}])^T$. The arithmetic expected per-round multiplier under fraction $f$ is
$$E[1 + f(R-1)] \;=\; 1 + f\bigl(E[R] - 1\bigr).$$
For this lottery $E[R] = 0.1 \cdot 10{,}001 + 0.9 \cdot 0 = 1000.1$, so the per-round multiplier is $1 + 999.1\,f$ and $E[W_T] = 1000 \cdot (1 + 999.1\,f)^{100}$.

Two extra diagnostics are crucial here, since most players end up with essentially nothing:

- **Survival rate** — fraction of players with $W_T > 0$.
- **Above-start rate** — fraction of players with $W_T > W_0$.

In [4]:
E_R = float(np.dot(probs, returns))   # arithmetic mean of R

def theoretical_E_WT(f):
    return initial_investment * (1 + f * (E_R - 1)) ** T

def diagnostics(final, f, label):
    return {
        "Strategy":            label,
        "Theoretical E[W_T]":  f"{theoretical_E_WT(f):.2e}",
        "Simulated E[W_T]":    f"{final.mean():.2e}",
        "Survival rate":       f"{(final > 0).mean():.4%}",
        "Above-start rate":    f"{(final > initial_investment).mean():.4%}",
    }

ev_table = pd.DataFrame([
    diagnostics(kelly_final, f_kelly, f"Kelly (f = {f_kelly})"),
    diagnostics(sbf_final,   f_sbf,   f"SBF (f = {f_sbf})"),
    diagnostics(rn_final,    f_rn,    f"Risk neutral (f = {f_rn})"),
])
print(ev_table.to_string(index=False))

              Strategy Theoretical E[W_T] Simulated E[W_T] Survival rate Above-start rate
    Kelly (f = 0.0999)          2.24e+203         5.46e+61     100.0000%         99.9800%
       SBF (f = 0.535)          7.54e+275         2.44e+55     100.0000%         67.9200%
Risk neutral (f = 1.0)          1.01e+303         0.00e+00       0.0000%          0.0000%


## Percentiles of the wealth distribution

In [5]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9, 99.99]

def fmt(x):
    return f"{x:.2e}" if x > 0 else "0"

table = pd.DataFrame({
    "Percentile":            percentiles,
    "Kelly":                 [fmt(v) for v in np.percentile(kelly_final, percentiles)],
    "SBF":                   [fmt(v) for v in np.percentile(sbf_final,   percentiles)],
    "Risk neutral":          [fmt(v) for v in np.percentile(rn_final,    percentiles)],
})
print(table.to_string(index=False))

 Percentile    Kelly      SBF Risk neutral
       1.00 4.09e+10 9.75e-15            0
       5.00 4.55e+13 1.12e-10            0
      10.00 5.05e+16 1.29e-06            0
      25.00 6.23e+22 1.71e+02            0
      50.00 7.69e+28 2.27e+10            0
      75.00 9.50e+34 3.00e+18            0
      90.00 1.17e+41 3.97e+26            0
      95.00 1.30e+44 4.57e+30            0
      99.00 1.79e+53 6.97e+42            0
      99.90 2.20e+59 9.22e+50            0
      99.99 2.72e+65 1.22e+59            0


## Top 10 final wealth values among the 10,000 players (each strategy)

In [6]:
kelly_top10 = np.sort(kelly_final)[::-1][:10]
sbf_top10   = np.sort(sbf_final)[::-1][:10]
rn_top10    = np.sort(rn_final)[::-1][:10]

top10_table = pd.DataFrame({
    "Rank":         np.arange(1, 11),
    "Kelly":        [fmt(v) for v in kelly_top10],
    "SBF":          [fmt(v) for v in sbf_top10],
    "Risk neutral": [fmt(v) for v in rn_top10],
})
print(top10_table.to_string(index=False))

 Rank    Kelly      SBF Risk neutral
    1 2.72e+65 1.22e+59            0
    2 2.72e+65 1.22e+59            0
    3 2.45e+62 1.06e+55            0
    4 2.45e+62 1.06e+55            0
    5 2.45e+62 1.06e+55            0
    6 2.45e+62 1.06e+55            0
    7 2.45e+62 1.06e+55            0
    8 2.45e+62 1.06e+55            0
    9 2.45e+62 1.06e+55            0
   10 2.20e+59 9.22e+50            0


**Takeaway.** This lottery is so favorable per round (per-dollar EV $\approx 999$) that even *very small* fractions can produce astronomical wealth — but the loss state wipes out everything you bet, and it hits 90% of the time. The three strategies illustrate how risk preferences interact with that ruin risk:

- **Risk neutral** ($f = 1$) maximizes theoretical $E[W_T]$ at $\sim 10^{303}$, but the only way to realize anything positive is to win all 100 rounds — probability $0.1^{100} \approx 10^{-100}$. With $N = 10{,}000$ players, *every single one* lost everything.
- **SBF** ($f \approx 0.535$) survives all 100 rounds with probability 1 in this simulation, but the median player ends with about $\$23$ billion which sounds great until you compare. Only ~68% of players end up *above their starting wealth*.
- **Kelly** ($f \approx 0.0999$) dominates SBF at every percentile shown — the median Kelly player ends at $\sim 10^{29}$ dollars (versus SBF's $\sim 10^{10}$), and 99.98% of Kelly players end up above start. The geometric/log-utility solution wins by a margin of $\sim$twenty orders of magnitude *at the median*.

The simulated mean ranking inverts the theoretical mean ranking: theoretically risk-neutral $\gg$ SBF $\gg$ Kelly; empirically Kelly $\gg$ SBF $\gg$ risk-neutral (= 0). With heavy-tailed lotteries and ruin risk, "$E[W_T]$" is a fantasy that no actual sample of players will witness.